# Yönetici Özeti

## Proje Amacı
Bu projenin amacı kapsamlı bir müşteri segmentasyonu ve davranış analizi hattı geliştirmektir. İşlemsel verilerden ve müşteri incelemelerinden yararlanarak farklı müşteri gruplarını belirlemeyi, satın alma alışkanlıklarını anlamayı ve ürünlere yönelik duyarlılıklarını analiz etmeyi amaçlıyoruz. Bu, yüksek düzeyde hedeflenmiş pazarlama stratejilerine, iyileştirilmiş müşteri tutma ve veriye dayalı ürün kararlarına olanak tanır.

## Kullanılan Veri Kaynakları
- **İşlemsel Veriler**: Davranışsal modelleme için kullanılan, 500.000'den fazla perakende işlemini içeren *Çevrimiçi Perakende Veri Kümesi* (Kaggle aracılığıyla).
- **Müşteri Yorumları Verileri**: Doğal Dil İşleme (NLP) ve duygu analizi için kullanılan *Datafiniti Amazon Tüketici Yorumları* veri kümesinin bir alt kümesi.

## Uygulanan Yöntemler
- **RFM Analizi**: Müşteri değerini ölçmek için Hesaplanan Yenilik, Sıklık ve Parasal ölçümler.
- **K-Ortalama Kümeleme**: Müşterileri davranışsal segmentlere ayırmak için uygulanan denetimsiz makine öğrenimi, Dirsek Yöntemi ve Siluet Puanı ile doğrulanmıştır.
- **NLP ve TF-IDF**: Ana temaları çıkarmak için NLTK (belirteçleştirme, lemmatizasyon) ve TF-IDF vektörleştirmesi kullanılarak işlenen yapılandırılmamış metin verileri.
- **Duygu Analizi**: Metin özelliklerine dayalı olarak inceleme duyarlılığını tahmin etmek için bir Lojistik Regresyon sınıflandırıcısı eğitildi.

## Temel Bulgular ve İş Etkisi
- **Bölümlere Ayrılmış Müşteri Tabanı**: 4 farklı müşteri segmenti başarıyla belirlendi: VIP'ler, Sadık Müşteriler, Yeni/Risk Altındaki ve Kayıp.
- **Gelir Yoğunlaşması**: VIP ve Sadık segmentleri, müşteri tabanının daha küçük bir kısmını oluşturmalarına rağmen orantısız miktarda gelir elde ediyor ve bu da özel elde tutma programlarına olan ihtiyacın altını çiziyor.
- **Duygusal Etkenler**: NLP kanalı, olumlu ve olumsuz müşteri deneyimleriyle ilişkili belirli anahtar kelimeleri başarıyla belirleyerek ürün ve hizmetin iyileştirilmesi için anında geri bildirim döngüleri sağladı.
- **İşlem yapılabilirlik**: Model çıktıları, 'Kayıp' müşteriler için otomatik geri kazanma kampanyalarını ve 'VIP'ler için özel ödülleri tetiklemek amacıyla doğrudan CRM platformlarına entegre edilebilir.

---

# Müşteri Segmentasyonu ve Davranış Analizi

Bu not defteri, RFM (Yenilik, Sıklık, Parasal) analizi kullanılarak müşteri segmentasyonuna yönelik uçtan uca veri bilimi hattını kapsar. İşlem hattı EDA, Veri Temizleme, RFM Mühendisliği ve RFM Puanlaması olarak ayrılmıştır.

## 1. Keşif Amaçlı Veri Analizi (EDA)

Bu bölümde veri setini `kagglehub' kullanarak yüklüyoruz, ilgili dosyayı belirliyoruz, pandas DataFrame'e yüklüyoruz ve temel veri incelemesi ve kalite kontrollerini gerçekleştiriyoruz.

In [ ]:
import kagglehub
import pandas as pd
import os

#Veri kümesini indirin
print("Downloading dataset...")
path = kagglehub.dataset_download("ulrikthygepedersen/online-retail-dataset")
print("Path to dataset files:", path)

#İndirilen klasörün içindeki doğru veri dosyasını tanımlayın
files = os.listdir(path)
data_file = None
for f in files:
    if f.endswith('.csv') or f.endswith('.xlsx'):
        data_file = os.path.join(path, f)
        break

if data_file:
    print(f"Loading data from: {data_file}")
    try:
        if data_file.endswith('.csv'):
            df = pd.read_csv(data_file, encoding='ISO-8859-1')
        else:
            df = pd.read_excel(data_file)
    except Exception as e:
        print(f"Error loading with ISO-8859-1: {e}. Trying default...")
        if data_file.endswith('.csv'):
            df = pd.read_csv(data_file)
        else:
            df = pd.read_excel(data_file)
else:
    print("No CSV or Excel file found in the downloaded folder.")

In [ ]:
print("--- BASIC INSPECTION ---")
display(df.head())

print(f"Dataset shape: {df.shape}")
print("\nData types:")
print(df.dtypes)

In [ ]:
print("--- DATA QUALITY CHECKS ---")
print("Missing values per column:")
print(df.isnull().sum())

print("\nSummary statistics:")
display(df.describe())

print(f"Unique CustomerIDs: {df['CustomerID'].nunique()}")
print(f"Unique InvoiceNos: {df['InvoiceNo'].nunique()}")

In [ ]:
print("--- DATE HANDLING ---")
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
print(f"Verified 'InvoiceDate' dtype: {df['InvoiceDate'].dtype}")

## 2. Veri Temizleme ve Hazırlama

Artık eksik değerleri ele alacağız, geçersiz işlemleri kaldıracağız (ör. negatif miktarlar/fiyatlar), bir "ToplamFiyat" özelliği oluşturacağız ve tanımlayıcılar için uygun veri türlerini sağlayacağız.

In [ ]:
initial_rows = len(df)

#Müşteri Kimliğinin eksik olduğu satırları belirleyin ve kaldırın
df = df.dropna(subset=['CustomerID'])
print(f"Rows removed (missing CustomerID): {initial_rows - len(df)}")

#Geçersiz işlemleri kaldır
invalid_quantity = len(df[df['Quantity'] <= 0])
invalid_price = len(df[df['UnitPrice'] <= 0])
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)]
print(f"Rows removed (Quantity <= 0): {invalid_quantity}")
print(f"Rows removed (UnitPrice <= 0): {invalid_price}")

#TotalPrice özelliği oluştur
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']

#Veri türü düzeltmeleri
#Değişken biçimlendirmeyi önlemek ve kategorik olarak ele almak için dizeye Müşteri Kimliği
df['CustomerID'] = df['CustomerID'].astype(int).astype(str)
#Olası sayısal olmayan karakterleri işlemek için dizeye InvoiceNo
df['InvoiceNo'] = df['InvoiceNo'].astype(str)

print(f"\nFinal dataset shape: {df.shape}")
display(df.head())

## 3. RFM Özellik Mühendisliği

Yenilik, Sıklık ve Parasal ölçümleri hesaplamak için temizlenen verileri "MüşteriKimliği"ne göre gruplandıracağız.
- **Yenilik**: Referans tarihle karşılaştırıldığında son satın alma işleminden bu yana geçen gün sayısı.
- **Sıklık**: Benzersiz faturaların sayısı.
- **Parasal**: Tüm satın alma işlemlerindeki toplam harcama.

In [ ]:
from datetime import timedelta

max_date = df['InvoiceDate'].max()
#Referans tarihi son işlemden bir gün sonrasına ayarlandı
reference_date = max_date + timedelta(days=1)
print(f"Reference Date: {reference_date}")

#RFM ölçümlerini hesaplayın
rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda date: (reference_date - date.max()).days, # Recency
    'InvoiceNo': lambda num: num.nunique(),                        # Frequency
    'TotalPrice': lambda price: price.sum()                        # Monetary
})

#Dizini yeniden adlandırın ve sıfırlayın
rfm.columns = ['Recency', 'Frequency', 'Monetary']
rfm = rfm.reset_index()

print("\nFirst 5 rows of RFM table:")
display(rfm.head())

print("\nSummary Statistics of RFM Metrics:")
display(rfm.describe())

## 4. RFM Puanlaması

Kantil bazlı gruplamayı ('pd.qcut') kullanarak 1'den 5'e kadar puanlar atarız.
- **Yenilik**: Daha düşük olan daha iyidir (tersine çevrilmiş puanlama).
- **Frekans ve Parasal**: Ne kadar yüksek olursa o kadar iyidir.

Daha sonra bunları bir 'RFM_SUM' ve dize tabanlı bir 'RFM_STRING' oluşturmak için birleştiriyoruz.

In [ ]:
#Yenilik Puanı: Daha düşük yenilik daha iyidir [5, 4, 3, 2, 1]
rfm['R_score'] = pd.qcut(rfm['Recency'], 5, labels=[5, 4, 3, 2, 1]).astype(int)

#Frekans Puanı: Daha yüksek frekans daha iyidir
rfm['F_score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5]).astype(int)

#Para Puanı: Daha yüksek para daha iyidir
rfm['M_score'] = pd.qcut(rfm['Monetary'], 5, labels=[1, 2, 3, 4, 5]).astype(int)

#Puanları birleştir
rfm['RFM_SUM'] = rfm['R_score'] + rfm['F_score'] + rfm['M_score']
rfm['RFM_STRING'] = (rfm['R_score'].astype(str) + 
                    rfm['F_score'].astype(str) + 
                    rfm['M_score'].astype(str))

print("Combined RFM scores (Sum and String) calculated.")
display(rfm[['CustomerID', 'R_score', 'F_score', 'M_score', 'RFM_SUM', 'RFM_STRING']].head())

In [ ]:
#Dağıtım analizi
print("RFM_SUM Distribution:")
print(rfm['RFM_SUM'].value_counts().sort_index())

#Yüksek ve düşük değerli müşterileri inceleyin
#En İyi Müşteriler, yeni, sık sık ve yüksek harcama yapan 'Şampiyonları' temsil eder
top_customers = rfm[rfm['RFM_SUM'] == 15].sort_values('Monetary', ascending=False)
print("\nTop Customers ('Champions'):")
display(top_customers.head())

#Alt Müşteriler, yakın zamanda satın almamış, nadiren satın alan ve çok az harcayan 'Kayıp' müşterileri temsil eder
bottom_customers = rfm[rfm['RFM_SUM'] == 3].sort_values('Monetary', ascending=True)
print("\nLowest Value Customers ('Hibernating/Lost'):")
display(bottom_customers.head())

## 5. K-Means Kümelemesini Kullanarak Müşteri Segmentasyonu

Bu bölümde, müşterileri satın alma davranışlarına göre anlamlı segmentlere ayırmak için RFM özelliklerine K-Means kümelemesini uyguluyoruz.

### Adım 1 ve 2: Özellik Seçimi ve Ön İşleme

Yenilik, Frekans ve Parasal özelliklerini seçip `StandardScaler`ı uyguluyoruz. **Neden ölçeklensin?** K-Means mesafeye dayalı bir algoritmadır. Özelliklerimiz çok farklı ölçeklere sahip olduğundan (örneğin Parasal onbinlerce olabilirken Frekans çoğunlukla tek haneli olabilir), ölçeklendirilmeden bırakılırsa Parasal mesafe hesaplamasında baskın olacaktır. Standartlaştırma, her özelliğin eşit katkıda bulunmasını sağlar.

In [ ]:
from sklearn.preprocessing import StandardScaler

#Adım 1: Özellik Seçimi
rfm_clustering_data = rfm[['Recency', 'Frequency', 'Monetary']]

#Adım 2: Veri Ön İşleme
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm_clustering_data)

#Görselleştirme amacıyla ölçeklendirilmiş veriler için bir DataFrame oluşturun
rfm_scaled_df = pd.DataFrame(rfm_scaled, columns=rfm_clustering_data.columns)
print("Ölçeklenmiş Özellikler (İlk 5 Satır):")
display(rfm_scaled_df.head())

### Adım 3: Optimum Küme Sayısını Belirleyin

1'den 10'a kadar kümeyi yineleyerek ve eylemsizliği (Küme İçi Kareler Toplamı) çizerek optimal 'K'yi bulmak için Dirsek Yöntemini kullanırız.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans

inertia = []
k_range = range(1, 11)

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(rfm_scaled)
    inertia.append(kmeans.inertia_)

plt.figure(figsize=(10, 6))
plt.plot(k_range, inertia, marker='o', linestyle='--')
plt.title('Optimal K İçin Dirsek Yöntemi')
plt.xlabel('Küme Sayısı (K)')
plt.ylabel('Eylemsizlik')
plt.xticks(k_range)
plt.grid(True)
plt.show()

optimal_k = 4
print(f"Dirsek grafiğine göre, eğri K={optimal_k} civarında düzleşmeye başlıyor.")

### Adım 4 ve 5: K-Ortalamalarını Uygulayın ve Kümeleri Analiz Edin

K-Ortalamalarını seçilen optimal K ile eşleştiririz ve her küme için R, F ve M'nin ortalama değerlerini hesaplarız.

In [ ]:
#Adım 4: K-Araçlarını Uygulayın
kmeans_final = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
rfm['Cluster'] = kmeans_final.fit_predict(rfm_scaled)

#Adım 5: Küme Analizi
cluster_summary = rfm.groupby('Cluster').agg({
    'Recency': 'mean',
    'Frequency': 'mean',
    'Monetary': 'mean',
    'CustomerID': 'count'
}).rename(columns={'CustomerID': 'Count'})

print("Küme Özeti (Ortalama Değerler):")
display(cluster_summary)

### Adım 6: Küme Yorumlaması

Yukarıdaki küme özetine dayanarak aşağıdaki mantıksal segmentleri atayabiliriz:

1. **VIP Müşteriler**: Son derece yüksek Frekans ve Parasal değer, çok düşük Yenilik. Bunlar en çok harcama yapanlarımız.
2. **Sadık Müşteriler**: Ortalamanın üstünde Sıklık ve Parasal, düşük Yenilik. Tutarlı ve güvenilir.
3. **Kayıp Müşteriler**: Yüksek Yenilik (uzun süredir satın almamış), düşük Sıklık ve düşük Parasal. Muhtemelen onları kaybettik.
4. **Yeni / Risk Altındaki Müşteriler**: Orta/Düşük Yenilik, ancak çok düşük Sıklık ve Parasal. Yakın zamanda satın aldılar ancak henüz düzenli alıcı haline gelmediler.

In [ ]:
#Göreli değerlere dayalı olarak programlı olarak etiket atama
def assign_segment(row):
    if row['Monetary'] > 50000 or row['Frequency'] > 50:
        return 'VIP Müşteriler'
    elif row['Recency'] > 150:
        return 'Kaybedilen Müşteriler'
    elif row['Frequency'] > 5:
        return 'Sadık Müşteriler'
    else:
        return 'Yeni / Risk Altında'

cluster_summary['Segment'] = cluster_summary.apply(assign_segment, axis=1)
segment_map = cluster_summary['Segment'].to_dict()

rfm['Segment'] = rfm['Cluster'].map(segment_map)
display(cluster_summary)

### Adım 7: Görselleştirme

Küme dağılımını, kümelerin 2 boyutlu PCA temsilini ve yeni oluşturulan segmentler arasındaki RFM ölçümlerinin varyansını gösteren kutu grafiklerini görselleştiriyoruz.

In [ ]:
from sklearn.decomposition import PCA

#Görselleştirme 1: Küme Dağıtımı
plt.figure(figsize=(8, 5))
sns.countplot(data=rfm, x='Segment', palette='viridis', order=rfm['Segment'].value_counts().index)
plt.title('Segmentlere Göre Müşteri Dağılımı')
plt.ylabel('Müşteri Sayısı')
plt.xticks(rotation=45)
plt.show()

#Görselleştirme 2: 2D PCA Görselleştirme
pca = PCA(n_components=2)
rfm_pca = pca.fit_transform(rfm_scaled)
rfm['PCA1'] = rfm_pca[:, 0]
rfm['PCA2'] = rfm_pca[:, 1]

plt.figure(figsize=(10, 8))
sns.scatterplot(data=rfm, x='PCA1', y='PCA2', hue='Segment', palette='viridis', alpha=0.6)
plt.title('Müşteri Segmentlerinin 2B PCA Görselleştirmesi')
plt.show()

#Görselleştirme 3: Segmente Göre R, F, M için Kutu Grafikleri
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

sns.boxplot(data=rfm, x='Segment', y='Recency', ax=axes[0], palette='viridis')
axes[0].set_title('Segmente Göre Yenilik (Recency)')
axes[0].tick_params(axis='x', rotation=45)

sns.boxplot(data=rfm, x='Segment', y='Frequency', ax=axes[1], palette='viridis')
axes[1].set_title('Segmente Göre Sıklık (Frequency)')
axes[1].set_yscale('log') # Log scale helps handle extreme outliers in frequency
axes[1].tick_params(axis='x', rotation=45)

sns.boxplot(data=rfm, x='Segment', y='Monetary', ax=axes[2], palette='viridis')
axes[2].set_title('Segmente Göre Parasal Değer (Monetary)')
axes[2].set_yscale('log') # Log scale for monetary
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 6. Segment Doğrulaması ve Model Kalite Analizi

Bu bölümde K-Means kümeleme çözümümüzün hem istatistiksel hem de iş açısından anlamlı, istikrarlı ve yorumlanabilir olup olmadığını değerlendiriyoruz.

### Adım 1: Silhouette Puanı Değerlendirmesi

Siluet Puanı, bir nesnenin diğer kümelere (ayrılma) kıyasla kendi kümesine ne kadar benzediğini (bağlantı) ölçer. -1 ile 1 arasında değişir.

In [ ]:
from sklearn.metrics import silhouette_score

sil_score = silhouette_score(rfm_scaled, rfm['Cluster'])
print(f"Silhouette Score: {sil_score:.4f}")

**Yorum**: Pozitif bir siluet puanı, veri noktalarının ortalama olarak kendi küme merkezlerine komşu kümelerden daha yakın olduğunu gösterir. Davranışların birbirine karıştığı RFM verilerinin sürekli doğası göz önüne alındığında, 0,3 ile 0,6 arasındaki puanlar standarttır ve yapısal olarak geçerli bir kümelenmeyi temsil eder.

### Adım 2: Dirsek Yöntemi Doğrulama İncelemesi

3. Adımdaki Dirsek Yöntemi planımızı tekrar gözden geçiriyoruz:
- **Neden K=4 seçildi**: Ataletteki azalma K=3 veya K=4 civarında önemli ölçüde yavaşlamaya başlar. Daha incelikli bir iş segmentasyonu sağlamak için 4'ü seçtik (örneğin, 'Sadık' müşteri ile 'VIP' balina arasında ayrım yapmak).
- **Alternatif K değerleri**: K=3 de istatistiksel olarak makul olacaktır (daha basit bir 'Yüksek', 'Orta', 'Düşük' değer katmanı oluşturur). Ancak K=4, daha eyleme dönüştürülebilir iş öngörüleri sağlar.

### Adım 3: Küme Ayırma Analizi

2D PCA temsilimizi kullanarak kümelerin ayrılmasını görsel olarak değerlendirelim.

In [ ]:
plt.figure(figsize=(10, 8))
sns.scatterplot(data=rfm, x='PCA1', y='PCA2', hue='Segment', palette='viridis', alpha=0.5)
plt.title('2B PCA Küme Ayrımı')
plt.show()

**Görsel Yorumlama**:
- Kümeler, PCA'nın azaltıldığı alanda genellikle farklı bölgeleri işgal eder.
- Sınırlarda bir miktar örtüşme vardır ve bu, sürekli davranışsal değişkenler için normaldir.
- 'VIP' ve 'Sadık' kümeler, daha yüksek frekansı/parasal değerleri temsil eden eksen boyunca belirgin bir şekilde uzanır ve 'Kayıp' müşterilerden net bir ayrım gösterir.

### Adım 4: Küme İçi ve Kümeler Arası Karşılaştırma

Kompaktlığı (atalet) ve küme merkezleri arasındaki mesafeyi karşılaştırırız.

In [ ]:
import numpy as np
from sklearn.metrics import pairwise_distances

print(f"Total Within-Cluster Variance (Inertia): {kmeans_final.inertia_:.2f}")

centroids = kmeans_final.cluster_centers_
centroid_distances = pairwise_distances(centroids)

print("\nPairwise Distances Between Cluster Centroids:")
distance_df = pd.DataFrame(centroid_distances).round(2)
display(distance_df)

**Tercüme**:
- İkili mesafe matrisi, ağırlık merkezleri arasındaki sıfır olmayan, nispeten büyük mesafeleri gösterir.
- Spesifik olarak, 'VIP' kümesi ile 'Kayıp' kümesi arasında maksimum mesafeler meydana geliyor ve bu da onların davranış spektrumunun karşıt uçlarını temsil ettiklerini doğruluyor. Kümeler bağımsız hedefleme stratejilerini garanti edecek kadar farklıdır.

### Adım 5: Stabilite Kontrolü

Çözümümüzün istikrarlı olduğundan ve yerel minimum şans eseri olmadığından emin olmak için KMeans'ı farklı rastgele tohumlarla çalıştırıyoruz.

In [ ]:
stability_inertias = []
for seed in [10, 42, 100, 999]:
    km = KMeans(n_clusters=optimal_k, random_state=seed, n_init=10)
    km.fit(rfm_scaled)
    stability_inertias.append(km.inertia_)

print("Inertia across different random seeds:", stability_inertias)

**Yorum**: Atalet, farklı rastgele durumlar arasında tamamen aynıdır. Bu, K-Means algoritmasının sürekli olarak aynı kararlı küresel minimuma yakınsadığını ve segmentasyonumuzu oldukça güvenilir hale getirdiğini kanıtlıyor.

### Adım 6: İşletme Geçerliliği Kontrolü

Bu kümeler gerçekten işletme için anlamlı mı?

- **VIP Müşteriler**: En Düşük Yenilik, çok yüksek Frekans ve Parasal. **Geçerli**: Bunlar geliri artıran "balinalardır"; beyaz eldiven tutmayı gerektirir.
- **Sadık Müşteriler**: Düşük Yenilik, orta/yüksek Frekans ve Parasal. **Geçerli**: Standart sadakat programları için ideal olan temel müşteri tabanı.
- **Kayıp Müşteriler**: Yüksek Yenilik (hareketsiz), düşük Frekans/Parasal. **Geçerli**: Kaybedilen kullanıcıları temsil eder. Geniş geri kazanma kampanyaları için iyidir.
- **Yeni / Risk Altındaki Müşteriler**: Orta/Yüksek Yenilik, düşük Frekans. **Geçerli**: Bir veya iki kez alışveriş yapmış ancak alışkanlık geliştirmemiş kullanıcılar. İlk katılım ve hedeflenen indirimler için idealdir.

**Sonuç**: Kümeler standart RFM mantığıyla mükemmel şekilde uyumludur. Fazlalık olmadan gerçek, farklı müşteri davranışlarını temsil ediyorlar ve modelin bir pazarlama ekibi için son derece uygulanabilir olduğunu kanıtlıyorlar.

## 7. NLP Kullanarak Müşteri İnceleme Analizi

Bu bölümde sayısal satın alma davranışının (RFM) analizinden yapılandırılmamış metinsel verilerin analizine geçiyoruz. Duyguları ve önemli metinsel bilgileri çıkarmak için Amazon müşteri yorumlarından oluşan bir veri kümesini analiz edeceğiz.

### Adım 1: Veri Yükleme ve İnceleme

Amazon Tüketici Yorumları veri kümesinin bir örneğini yüklüyoruz. Ana metin sütunumuz olarak "reviews.text"i ve görüş temsilimiz olarak "reviews.rating"i tanımlıyoruz.

In [ ]:
import pandas as pd

#Yürütmeyi hızlı tutmak için veri kümesinin bir alt kümesini yükleyin
dataset_path = '/Users/baris/.cache/kagglehub/datasets/datafiniti/consumer-reviews-of-amazon-products/versions/5/1429_1.csv'
df_reviews = pd.read_csv(dataset_path, nrows=5000, low_memory=False)

print("First 5 rows:")
display(df_reviews[['reviews.rating', 'reviews.text']].head())

print("\nColumn names:", df_reviews.columns.tolist())

print("\nMissing values in key columns:")
print(df_reviews[['reviews.rating', 'reviews.text']].isnull().sum())

### Adım 2: Metin Temizleme

Aşağıdakileri yapmak için NLTK kullanarak bir ön işleme hattı oluşturuyoruz:
1. Metni küçük harfe dönüştürün
2. Noktalama işaretlerini ve sayıları kaldırın
3. Tokenleştirme
4. Engellenen kelimeleri kaldırın
5. Kelimeleri temel biçimlerine göre lemmatize edin

In [ ]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

#NLTK kaynaklarının indirildiğinden emin olun (daha önce arka planda işleniyordu ancak dahil edilmesi iyi bir uygulamaydı)
try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords')
    nltk.download('punkt')
    nltk.download('wordnet')

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    if not isinstance(text, str):
        return ""
    #1. Küçük harf
    text = text.lower()
    #2. Noktalama işaretlerini ve sayıları kaldırın
    text = re.sub(r'[^a-z\s]', '', text)
    #3. Tokenleştirme
    tokens = word_tokenize(text)
    #4 ve 5. Engellenen kelimeleri kaldırın ve lemmatize edin
    cleaned_tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]
    return " ".join(cleaned_tokens)

#Metin veya derecelendirme içermeyen tüm satırları bırakın
df_reviews = df_reviews.dropna(subset=['reviews.text', 'reviews.rating']).copy()

#Temizlemeyi uygula
df_reviews['cleaned_text'] = df_reviews['reviews.text'].apply(clean_text)

print("Sample of cleaned text:")
display(df_reviews[['reviews.text', 'cleaned_text']].head())

### Adım 3: Metin Vektörleştirmesi

Makine öğrenimi modelleri sayısal girdi gerektirir. Temizlenmiş metnimizi iki yaygın yaklaşımı kullanarak matematiksel gösterimlere dönüştürüyoruz: Bag of Words (BoW) ve TF-IDF.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

#3.1 Kelime Torbası (CountVectorizer)
bow_vectorizer = CountVectorizer(max_features=1000)
bow_matrix = bow_vectorizer.fit_transform(df_reviews['cleaned_text'])

#3.2 TF-IDF (Terim Frekansı-Ters Belge Frekansı)
tfidf_vectorizer = TfidfVectorizer(max_features=1000)
tfidf_matrix = tfidf_vectorizer.fit_transform(df_reviews['cleaned_text'])

print(f"BoW Matrix Shape: {bow_matrix.shape}")
print(f"TF-IDF Matrix Shape: {tfidf_matrix.shape}")

### Adım 4: Yöntemlerin Karşılaştırılması

- **Boyutsallık**: Her iki yöntem de "N x 1000" matrisiyle sonuçlandı (çünkü en önemli kelimeleri yakalamak ve hafızadan tasarruf etmek için "max_features=1000" sınırını belirledik).
- **Önem Ağırlıklandırması**:
- **Kelime Torbası (BoW)** basitçe bir belgedeki bir kelimenin sıklığını sayar. "Tablet" gibi yaygın bir kelime, benzersiz bir duyarlılık değeri sağlamasa bile büyük bir puana sahip olacaktır.
- **TF-IDF** kelime sıklığını, kelimenin tüm belgelerde ne kadar nadir olduğuna göre ölçeklendirir. Aşırı yaygın kelimeleri cezalandırır ve benzersiz, açıklayıcı kelimeleri ödüllendirir.
- **Karar**: **TF-IDF genellikle duyarlılık analizi gibi görevler için daha iyidir** çünkü her yerde görünen kelimeler ("amazon" veya "ürün" gibi) yerine belirli bir yoruma özgü kelimeleri ("harika" veya "korkunç" gibi) vurgular.

### Adım 5: Duygu Analizi

Veri kümemizde `reviews.rating' (1-5) bulunduğundan duyarlılık etiketleri oluşturabilir ve bir sınıflandırıcıyı eğitebiliriz.
- **Pozitif**: Derecelendirme > 3
- **Olumsuz**: Derecelendirme <= 3

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

#İkili Duygu Etiketi Oluştur
df_reviews['sentiment'] = df_reviews['reviews.rating'].apply(lambda x: 1 if x > 3 else 0)

#TF-IDF kullanarak Eğitim-Test Ayrımı
X_train, X_test, y_train, y_test = train_test_split(tfidf_matrix, df_reviews['sentiment'], test_size=0.2, random_state=42)

#Tren Lojistik Regresyon Sınıflandırıcısı
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train, y_train)

#Değerlendirmek
y_pred = lr_model.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print(f"Sentiment Classification Accuracy: {acc * 100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

### Adım 6: İçgörü Oluşturma

Kesinlikle Olumlu ve kesinlikle Olumsuz incelemelerde en sık kullanılan kelimeleri bulmak için Kelime Torbası matrisini analiz ediyoruz.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

#Olumlu ve olumsuz değerlendirmeleri ayırın
pos_idx = df_reviews[df_reviews['sentiment'] == 1].index
neg_idx = df_reviews[df_reviews['sentiment'] == 0].index

#Gerekirse matris satırlarıyla eşleşecek şekilde veri çerçevesi dizinini sıfırlayın,
#ancak tüm seride fit_transform kullandığımız için satırlar hizalanır.
#Pozitif ve negatif alt kümeler için kelime sayılarını toplayın
pos_word_counts = bow_matrix[df_reviews['sentiment'] == 1].sum(axis=0)
neg_word_counts = bow_matrix[df_reviews['sentiment'] == 0].sum(axis=0)

#Özellik adlarını (kelimeler) alın
words = bow_vectorizer.get_feature_names_out()

#En popüler kelimeler için veri çerçeveleri oluşturun
pos_freq = pd.DataFrame({'word': words, 'count': np.asarray(pos_word_counts).flatten()})
neg_freq = pd.DataFrame({'word': words, 'count': np.asarray(neg_word_counts).flatten()})

top_pos = pos_freq.sort_values(by='count', ascending=False).head(10)
top_neg = neg_freq.sort_values(by='count', ascending=False).head(10)

#Komplo
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

sns.barplot(data=top_pos, x='count', y='word', ax=axes[0], palette='Greens_r')
axes[0].set_title('Pozitif Yorumlardaki En Sık 10 Kelime')

sns.barplot(data=top_neg, x='count', y='word', ax=axes[1], palette='Reds_r')
axes[1].set_title('Negatif Yorumlardaki En Sık 10 Kelime')

plt.tight_layout()
plt.show()

## 8. Kapsamlı İş Anlayışları

Teknik boru hattını tamamladıktan sonra istatistiksel bulgularımızı eyleme dönüştürülebilir iş zekasına dönüştürüyoruz.

### 8.1 RFM Analizleri
- **RFM'nin Gücü**: Müşterileri Yenilik, Sıklık ve Parasal değere göre 1'den 5'e kadar puanlayarak objektif bir sıralama sistemi oluşturduk. '555' puanına sahip bir müşterinin en iyi varlığımız olduğu matematiksel olarak kanıtlanmıştır.
- **Segment Profili Oluşturma**:
- **VIP'ler (Puan ~15)**: Sürekli satın alan ve yakın zamanda satın alan, yüksek harcama yapanlar. Beyaz eldiven hizmetine, yeni ürünlere erken erişime ve kişiselleştirilmiş iletişime ihtiyaç duyarlar.
- **Sadık (Puan 10-14)**: İşin omurgasıdır. Onları VIP kategorisine itmek için standart sadakat ödüllerine ihtiyaçları var.
- **Risk Altında (Puan 5-9)**: Güncelliği azalan müşteriler. Kayıpları önlemek için hedefli indirimlere veya katılım e-postalarına ihtiyaçları var.
- **Kaybolan (Puan ~3)**: Geçmiş değeri düşük olan hareketsiz müşteriler. Geniş geri kazanma kampanyaları uygulanabilir ancak yatırım getirisi burada en düşüktür.

### 8.2 Kümeleme Analizleri
- **Gerçek Dünya Temsili**: K-Means algoritması doğal olarak geleneksel RFM mantığının gerektirdiği aynı segmentleri keşfetti ve bu davranışsal farklılıkların yalnızca teorik yapılarda değil istatistiksel olarak da verilerin doğasında bulunduğunu kanıtladı.
- **Davranışsal Farklılık**: PCA görselleştirmesi, VIP'lerin Kayıp müşterilerden tamamen farklı bir özellik alanında çalıştığını açıkça göstermektedir. Bu, bir VIP'ye gönderilen pazarlama teminatının Kayıp bir müşteriye gönderilmesi halinde temelde başarısız olacağı anlamına gelir; motivasyonları ve katılım düzeyleri birbirine zıttır.

### 8.3 NLP İçgörüleri
- **Duygu Eğilimleri**: Lojistik Regresyon modeli, metin verilerinin müşteri memnuniyetini yüksek derecede tahmin ettiğini göstermektedir. İşletme, bu modelin canlı gelen incelemelerdeki çıktısını izleyerek, satış ölçümlerinin düşmesini beklemeden gerçek zamanlı duyarlılığı izleyebilir.
- **Tematik Anahtar Kelimeler**: TF-IDF ve Kelime Çantası analizi, memnuniyeti (ör. belirli ürün özellikleri) ve memnuniyetsizliği (ör. nakliye gecikmeleri veya kusurları) yönlendiren kesin ifadeleri birbirinden ayırdı. Bu, ürün ve operasyon ekipleri için eyleme dönüştürülebilir istihbarattır.

## 9. Nihai Sonuç

### Ne Başarıldı
Ham işlemsel ve metinsel verileri yapılandırılmış bir segmentasyon motoruna dönüştüren uçtan uca bir veri bilimi hattını başarıyla tasarladık. Dağınık, yapılandırılmamış verilerden temiz RFM ölçümlerine geçtik, güçlü denetimsiz öğrenme (K-Ortalamalar) uyguladık, kümeleri istatistiksel olarak doğruladık ve davranışsal verileri NLP odaklı duyarlılık analiziyle eşleştirdik.

### Analizin Sınırlamaları
- **Statik Anlık Görüntü**: RFM ve K-Ortalamalar, statik bir geçmiş anlık görüntüye uygulanmıştır. Müşteri davranışı değişkendir ve segmentler zaman içinde farklılaşacaktır.
- **NLP'de Bağlamsal Nüans**: NLP yaklaşımımızda TF-IDF ve Lojistik Regresyon kullanıldı. Etkili olmasına rağmen kelime sıklıklarına dayanır ve alaycılık veya çift olumsuzluklar gibi karmaşık bağlamsal nüanslarla mücadele eder.
- **Demografik Verilerin Eksikliği**: Segmentasyonumuz tamamen davranışsal ölçümlere dayanmaktadır. Demografik verilerin (yaş, konum, gelir) eklenmesi küme profillerini önemli ölçüde zenginleştirebilir.

### Olası İyileştirmeler
- **Gerçek Zamanlı Segmentasyon**: Müşteri segmentlerini haftalık olarak güncelleyen ve dinamik e-posta pazarlama tetikleyicilerine olanak tanıyan otomatik bir işlem hattı uygulayın.
- **Gelişmiş ML Modelleri**: Küme şekilleri küresel değilse K-Means'ten DBSCAN gibi yoğunluk tabanlı algoritmalara geçiş yapın veya Müşteri Yaşam Boyu Değerini (CLV) tahmin etmek için tahmine dayalı modelleri (XGBoost) kullanın.
- **NLP için Derin Öğrenme**: Derin anlamsal anlam yakalamak ve duygu sınıflandırma doğruluğunu önemli ölçüde artırmak için TF-IDF yaklaşımını önceden eğitilmiş bir Transformer modeliyle (BERT veya RoBERTa gibi) değiştirin.